In [1]:
# Import pandas for data manipulation and analysis (works with tabular data like CSVs)
import pandas as pd

# Import numpy for numerical operations (arrays, math functions, etc.)
import numpy as np

# Import itertools for advanced iteration tools (combinations, permutations, etc.)
import itertools

# Import text vectorizers from scikit-learn:
# - CountVectorizer: converts text into word counts
# - TfidfVectorizer: converts text into TF-IDF scores (importance of words)
# - HashingVectorizer: converts text into fixed-length feature vectors using hashing
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer, HashingVectorizer

# Import train_test_split to split dataset into training and testing sets
from sklearn.model_selection import train_test_split

# Import PassiveAggressiveClassifier (a linear model useful for large-scale text classification)
from sklearn.linear_model import PassiveAggressiveClassifier

# Import Multinomial Naive Bayes (commonly used for text classification with word counts)
from sklearn.naive_bayes import MultinomialNB

# Import metrics for evaluating model performance (accuracy, confusion matrix, etc.)
from sklearn import metrics

# Import matplotlib for plotting graphs and visualizations
import matplotlib.pyplot as plt


In [3]:
# Read the CSV file named 'news.csv' into a pandas DataFrame and instruct pandas not to assign any column as index
df = pd.read_csv('news.csv', index_col=None)

In [4]:
print(df.head())

   Unnamed: 0                                              title  \
0        8476                       You Can Smell Hillary’s Fear   
1       10294  Watch The Exact Moment Paul Ryan Committed Pol...   
2        3608        Kerry to go to Paris in gesture of sympathy   
3       10142  Bernie supporters on Twitter erupt in anger ag...   
4         875   The Battle of New York: Why This Primary Matters   

                                                text label  
0  Daniel Greenfield, a Shillman Journalism Fello...  FAKE  
1  Google Pinterest Digg Linkedin Reddit Stumbleu...  FAKE  
2  U.S. Secretary of State John F. Kerry said Mon...  REAL  
3  — Kaydee King (@KaydeeKing) November 9, 2016 T...  FAKE  
4  It's primary day in New York and front-runners...  REAL  


In [5]:
dataset=df.drop("Unnamed: 0", axis=1)

In [6]:
print(dataset.head())

                                               title  \
0                       You Can Smell Hillary’s Fear   
1  Watch The Exact Moment Paul Ryan Committed Pol...   
2        Kerry to go to Paris in gesture of sympathy   
3  Bernie supporters on Twitter erupt in anger ag...   
4   The Battle of New York: Why This Primary Matters   

                                                text label  
0  Daniel Greenfield, a Shillman Journalism Fello...  FAKE  
1  Google Pinterest Digg Linkedin Reddit Stumbleu...  FAKE  
2  U.S. Secretary of State John F. Kerry said Mon...  REAL  
3  — Kaydee King (@KaydeeKing) November 9, 2016 T...  FAKE  
4  It's primary day in New York and front-runners...  REAL  


In [7]:
y=dataset["label"]
X_train, X_test, y_train, y_test = train_test_split(dataset['text'], y, test_size=0.33, random_state=53)

In [14]:
# - CountVectorizer: transforms text documents into a matrix of word counts
# - stop_words='english': removes common English words (like "the", "is", "and") 
counter_Vectorizer = CountVectorizer(stop_words='english')
counter_Train = counter_Vectorizer.fit_transform(X_train)
print(counter_Train)
count_test = counter_Vectorizer.transform(X_test)

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 1119820 stored elements and shape (4244, 56922)>
  Coords	Values
  (1, 42470)	1
  (1, 12105)	1
  (1, 54177)	1
  (1, 50628)	1
  (1, 15924)	2
  (1, 44520)	2
  (1, 51896)	2
  (1, 35783)	4
  (1, 35256)	1
  (1, 21881)	1
  (1, 42534)	1
  (1, 8399)	1
  (1, 29531)	2
  (1, 15927)	2
  (1, 25686)	1
  (1, 49203)	2
  (1, 16814)	1
  (1, 36087)	1
  (1, 21568)	1
  (1, 25684)	1
  (1, 38823)	1
  (1, 47506)	1
  (1, 36831)	1
  (2, 16972)	1
  (2, 762)	1
  :	:
  (4243, 41435)	1
  (4243, 53607)	1
  (4243, 659)	1
  (4243, 38834)	1
  (4243, 19003)	1
  (4243, 11415)	1
  (4243, 7545)	1
  (4243, 22426)	1
  (4243, 54007)	1
  (4243, 7113)	1
  (4243, 4932)	1
  (4243, 39497)	1
  (4243, 50053)	1
  (4243, 38849)	1
  (4243, 20702)	1
  (4243, 42139)	1
  (4243, 17247)	1
  (4243, 50052)	1
  (4243, 55228)	1
  (4243, 29255)	1
  (4243, 49435)	1
  (4243, 11257)	1
  (4243, 52945)	1
  (4243, 20905)	1
  (4243, 7962)	1


In [15]:
# Get the number of unique features (words) learned by CountVectorizer
# - count_vectorizer.get_feature_names_out(): returns an array of all words in the vocabulary
#   → This equals the number of features (columns) in your training matrix
len(counter_Vectorizer.get_feature_names_out())

56922

In [12]:
print(counter_Train.toarray())

[[0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 ...
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]]


In [16]:
# Step 1: Initialize the Multinomial Naive Bayes classifier
# - MultinomialNB is well-suited for text classification problems
#   where features are word counts or frequencies.
clf = MultinomialNB()

# Step 2: Train (fit) the classifier on the training data
# - count_train: the bag-of-words matrix for training documents
# - y_train: the true labels (e.g., FAKE or REAL) for training data
clf.fit(counter_Train, y_train)

# Step 3: Predict labels for the test data
# - count_test: the bag-of-words matrix for test documents
# - pred: the predicted labels for each test document
pred = clf.predict(count_test)

# Step 4: Calculate accuracy score
# - Compares predicted labels (pred) with true labels (y_test)
# - Returns the proportion of correct predictions
score = metrics.accuracy_score(y_test, pred)

# Step 5: Print accuracy with 3 decimal places
print("accuracy:   %0.3f" % score)

# Step 6: Generate confusion matrix
# - Compares true labels (y_test) with predicted labels (pred)
# - labels=['FAKE','REAL']: ensures matrix rows/columns are ordered
#   consistently for FAKE and REAL classes
cm = metrics.confusion_matrix(y_test, pred, labels=['FAKE', 'REAL'])


accuracy:   0.893


In [17]:
from sklearn.metrics import classification_report

report=classification_report(y_test, pred)

In [18]:
print(report)

              precision    recall  f1-score   support

        FAKE       0.92      0.86      0.89      1008
        REAL       0.88      0.93      0.90      1083

    accuracy                           0.89      2091
   macro avg       0.90      0.89      0.89      2091
weighted avg       0.89      0.89      0.89      2091



In [19]:
dataset["text"][0]

'Daniel Greenfield, a Shillman Journalism Fellow at the Freedom Center, is a New York writer focusing on radical Islam. \nIn the final stretch of the election, Hillary Rodham Clinton has gone to war with the FBI. \nThe word “unprecedented” has been thrown around so often this election that it ought to be retired. But it’s still unprecedented for the nominee of a major political party to go war with the FBI. \nBut that’s exactly what Hillary and her people have done. Coma patients just waking up now and watching an hour of CNN from their hospital beds would assume that FBI Director James Comey is Hillary’s opponent in this election. \nThe FBI is under attack by everyone from Obama to CNN. Hillary’s people have circulated a letter attacking Comey. There are currently more media hit pieces lambasting him than targeting Trump. It wouldn’t be too surprising if the Clintons or their allies were to start running attack ads against the FBI. \nThe FBI’s leadership is being warned that the entir

In [21]:
print(counter_Train[[0]])
print(X_train[[0]])
clf.predict(counter_Train[[0]])

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 0 stored elements and shape (1, 56922)>
0    Daniel Greenfield, a Shillman Journalism Fello...
Name: text, dtype: object


array(['FAKE'], dtype='<U4')